# Objektspårning och klassificering med AI 🎯

Du har tränat CNN:er och kört en live-kamera. Nu lägger vi till ett nytt lager: **objektspårning**.

## Tre lager i pipelinen

| Steg | Vad det gör | Vem gör det |
|------|-------------|-------------|
| **Detektion** | Hittar objekt i *ett frame* — ger bounding boxes | YOLOv8 |
| **Tracking** | Kopplar ihop boxes *över tid* — samma objekt, samma ID | ByteTrack |
| **Klassificering** | Vad exakt *är* det? — dina egna kategorier | MobileNetV2 / CLIP |

## Vad du behöver
- Ett Google-konto med Drive
- En kamera **eller** en videofil att ladda upp
- Kör alla celler uppifrån och ned

## Vad du väljer
- **Grunduppgift:** Träna en egen klassificerare (MobileNetV2) och plugga in den
- **Utmaningsuppgift:** Testa zero-shot klassificering med CLIP — inga träningsbilder krävs

---
## Del 1: Installation och laddning

Kör cellen nedan för att installera nödvändiga bibliotek. Det tar ~1 minut.

In [ ]:
!pip install ultralytics trackers supervision --quiet

In [ ]:
import os
import base64
import json
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

import supervision as sv
from ultralytics import YOLO
from trackers import ByteTrackTracker

from IPython.display import display, HTML
from google.colab import files, output as colab_output

print("Alla bibliotek importerade! ✓")

### Ladda detektor och tracker

YOLOv8n är den minsta och snabbaste varianten — bra för Colab.
Den är förtränad på 80 vanliga objekt (COCO-dataset): person, flaska, stol, laptop, bil...

In [ ]:
# Ladda YOLOv8n — laddas ner automatiskt första gången (~6 MB)
yolo = YOLO('yolov8n.pt')

# Skapa tracker — håller reda på vilka objekt som är vilka över tid
tracker = ByteTrackTracker()

# Annotators från supervision
box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.6)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=40)

# Klassnamn som YOLO känner till
YOLO_KLASSER = yolo.model.names  # dict: {0: 'person', 1: 'bicycle', ...}

print(f"YOLO laddad ✓  ({len(YOLO_KLASSER)} klasser)")
print(f"Exempel på klasser YOLO känner igen:")
print(", ".join(list(YOLO_KLASSER.values())[:20]))